In [ ]:
from tools.utils import *
from time import sleep
from tqdm import tqdm
from ollama import Client
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import torch
import spacy
import json

In [ ]:


with open('api_key.json') as f:
    api_key = json.load(f)['api_key']

client = Client(
            host="https://ollama.com",
            headers={'Authorization': api_key}
        )

def prompt_ollama_turbo(client,messages, model):
    try:
        return client.chat(model, messages=messages)
    except Exception as e:
        try:
            sleep(.5)
            return client.chat(model, messages=messages)
        except Exception as e:
            print(f"Error creating client: {e}")
            sleep(.5)
            return None
    


In [ ]:
definitions = {
    'class waiver' : """A class action waiver is a provision found in some contracts which prohibits a party from filing a class action legal proceeding against the other party, or both parties waiving the right to file class actions against each other. Most class action waiver clauses include this wording or a variation of it: You and we agree that any dispute filed against each other must be on an individual basis and not as a class or collective action.""",
    'opt-out' : """a clause that permits signatories to a contract to opt out of particular provisions, or to terminate the contract early""",
    'arbitration': """In contract law, an arbitration clause is a clause in a contract that requires the parties to resolve their disputes through an arbitration process. Although such a clause may or may not specify that arbitration occur within a specific jurisdiction, it always binds the parties to a type of resolution outside the courts, and is therefore considered a kind of forum selection clause.""",
    'modification': """This clause gives the platform rights to unilaterally change the contract at any time and states how an agreement can be changed or modified.""",
    'anti-scraping': """An anti-scraping clause is a provision in a contract that prohibits the use of automated tools or software to extract data from a website or online service without permission from the website owner. This clause is often included in the terms of service or user agreements of websites to protect their content and data from being harvested by third parties.""",
}

# Prepare data for annotation

In [ ]:
# # prepare the spacy model and pipeline
# nlp = spacy.load("en_core_web_sm")
# nlp.disable_pipes("tagger", "parser", "attribute_ruler", "lemmatizer")
# nlp.add_pipe('sentencizer')


In [ ]:
processed_data_dir = Path('processed_data/tous')

In [ ]:
embeddings = np.loadtxt(processed_data_dir / 'embedding.tsv')
metadata = pd.read_csv(processed_data_dir / 'metadata.tsv',sep='\t')
print(len(embeddings), len(metadata))

# Clause classification

In [ ]:
clause_type = 'modification' # 'modification' |'opt-out'  |'arbitration' | 'opt-out' | 'class waiver' | 'anti-scraping'
examples_df = pd.read_excel(f'annotations/seed_examples/{clause_type}_clauses.xlsx', sheet_name='Sheet1')

In [ ]:
examples_df['processed_text'] = examples_df['Examples'].apply(lambda x: 'clustering: ' + x.lower().strip().replace('\n', ' '))

In [ ]:
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True) # trust_remote_code is needed to use the encode method


In [ ]:
target_embeddings = model.encode(examples_df['processed_text'].tolist())

In [ ]:
average_embedding = target_embeddings.mean(axis=0)

In [ ]:
similarity_scores_max = cosine_similarity(embeddings, target_embeddings)
similarity_scores_max = similarity_scores_max.max(axis=1)
top_n_position_max = np.argpartition(similarity_scores_max, -500)[-500:][::-1]
np.random.seed(42)
dist_sample_max = np.random.choice(list(range(len(metadata))), p=(similarity_scores_max / similarity_scores_max.sum()), size=500, replace=False)

In [ ]:
similarity_scores_av = cosine_similarity(embeddings, [average_embedding]).flatten()
top_n_position_av = np.argpartition(similarity_scores_av, -500)[-500:][::-1]
np.random.seed(42)
dist_sample_av = np.random.choice(list(range(len(metadata))), p=(similarity_scores_av / similarity_scores_av.sum()), size=500, replace=False)

In [ ]:
sentences_positions = list(set(np.concatenate([top_n_position_max, top_n_position_av, dist_sample_max, dist_sample_av])))
print(f'Total sentences selected: {len(sentences_positions)}')

In [ ]:


out_csv = pd.DataFrame(metadata.iloc[sentences_positions].sentence.unique())
out_csv.columns = ['sentence']
out_csv['label'] = 0
#out_csv['similarity'] = similarity_scores[top_n_position]
out_csv.to_csv(f'annotations/similar_examples/{clause_type}_selected_sentences.csv')

## Pre-annotate selected examples with Ollama

In [ ]:
# prepare examples for annotation
df = pd.read_csv(f'annotations/similar_examples/{clause_type}_selected_sentences.csv', index_col=0)
df.columns


In [ ]:
df['prompt'] = df['sentence'].apply(lambda s: """
You are a helpful AI that classifies sentences as indicating the presence of a {0} clause (or not). 

We first provide you with a definition of a {0} clause followed by three snippets of {0} clauses. 
          
DEFINITIbeON: 
The definition of of {0} clause is: {3}

EXAMPLES:          
Below are three examples of {0} clauses.\n\n{1}
          
QUESTION:
Does the following sentence indicate the presence of a {0} clause? It does not need to be the complete clause, it can be part of the clause or contain an indication that the text contains a {0} clause.

Please reply only 'yes' or 'no', nothing else.
{2}
""".format(
     clause_type,examples_df['Examples'].sample(3).str.cat(sep='\n'),s, definitions[clause_type]))


In [ ]:
model_list = ["mistral-large-3:675b-cloud", "gpt-oss:120b", "deepseek-v3.1:671b-cloud"]

In [ ]:
for model in model_list:
    df[f'response_{model}'] = None

In [ ]:
tqdm.pandas()
for model in model_list:
    df[f'response_{model}'] = df.progress_apply(
            lambda row: prompt_ollama_turbo(client,[{"role": "user", "content": row['prompt']}], model=model) if pd.isna(row[f'response_{model}']) else row[f'response_{model}'], axis=1
        )   

In [ ]:
for model in model_list:
    df[f'label_{model}'] = df[f'response_{model}'].apply(lambda x: 1 if x.message.content.lower().strip() in ['yes', 'yes.'] else 0 if x.message.content.lower().strip() in ['no', 'no.'] else None)
    df[f'label_{model}'].value_counts()

In [ ]:
df['sum'] = df[[c for c in df.columns if c.startswith('label_')]].sum(axis=1)
df[['sentence', 'sum'] + [c for c in df.columns if c.startswith('label_')]].to_csv(f'annotations/ollama_annotations/{clause_type}_labels.csv')
df.to_csv(f'annotations/ollama_annotations/{clause_type}_annotations.csv')

# Fin.